In [0]:
%pip install -U -qqq mlflow-skinny[databricks] langgraph==0.3.4 databricks-langchain databricks-agents uv 
dbutils.library.restartPython()


# Generate synthetic data

Useing Mosaic AI Agent Evaluation, we can automatically generate a representative [evaluation set](https://docs.databricks.com/aws/en/generative-ai/agent-evaluation/synthesize-evaluation-set) from your documents, allowing you to quickly evaluate your agent with good coverage of test cases.

To synthesise evaluations for an agent that uses document retrieval, use the `generate_evals_df` method from the `databricks-agents` Python package. 

- **Input**: A table of documents (`content`, `doc_uri`) and evaluation guidelines
- **Output**: A generated set of eval questions stored as a Delta table
- **Purpose**: To test and refine retrieval performance using realistic, autogenerated examples


In [0]:
import pandas as pd
import mlflow

#Define the catalog name for each user 
# workspace_url = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
# digits = ''.join(filter(str.isdigit, workspace_url))
# catalog = 'catalog_' + digits

catalog = "ananyaroy"
schema = "agent_workshop_tabcorp"

exp_name = '/Users/ananya.roy@databricks.com/aws-db-agent-workshop/workshop-code/03-langGraph-create-register-deploy-tool-calling-agent'

mlflow.set_experiment(exp_name)


num_evals = 20 
model_version = 1

table_name = f"{catalog}.{schema}.guidance_unstructure"
eval_table = f"{catalog}.{schema}.finOps_synthetic_evals"
model_name = f"models:/{catalog}.{schema}.financial_assistance_agent/{model_version}"

# FinOps portfolio analyst persona , guideline for synthetic data generation 
guidelines = f"""
# Task Description
The agent is a financial portfolio advisor and a fraud assistant agent. The agents task is to provide guidance on difference scenarios of rebalancing , portfolio assessment , fraud analysis guidance , procedures of next steps if fraud happens etc.s

# User personas
- A customer who wants to assess and understand their portfolio and seek guidance on management.
- A customer who wants to understand the steps to take if fraud happens.
- A customer who wants to get guidance on portfolio rebalancing.

# Example questions
- What is the standard procedure for rebalancing my portfolio and what should i consider?
- How much time does it take for a fraud incident to resolve ?

# Additional Guidelines
- Questions should be succinct, and human-like
"""

In [0]:
# Use the synthetic eval generation API to get some evals
from databricks.agents.evals import generate_evals_df

parsed_docs_df = spark.table(table_name).withColumnRenamed('file_name', 'doc_uri').withColumnRenamed('file_content', 'content').toPandas()

evals = generate_evals_df(
    docs=parsed_docs_df, # Pass your docs. Pandas/Spark Dataframes with columns `content STRING, doc_uri STRING` are suitable.
    num_evals=num_evals, # How many synthetic evaluations to generate
    question_guidelines=guidelines
)
display(evals)

#store the synthetic evaluation set into a delta table
spark.createDataFrame(evals).write.format("delta").mode("overwrite").saveAsTable(eval_table)


## Generate Traces

In [0]:
# outside of predict function (loading model only once)
loaded_model = mlflow.pyfunc.load_model('models:/ananyaroy.agent_workshop_tabcorp.financial_assistance_agent/2')

question_list = evals['inputs'].tolist()
n = 1 
#Generate the set number of traces from the question list 
with mlflow.start_run(run_name=f'finance_multi_agent_offline_eval_{n}') as eval:
    # Simulate beta testing traffic with scenarios designed to fail guidelines
    # Run requests and capture traces
    for question in question_list:
        response = loaded_model.predict(question) 


# Offline Evaluation via Mosaic AI Evaluation
With Databricks, [agent evaluation](https://docs.databricks.com/aws/en/generative-ai/agent-evaluation/) helps us to evaluate the quality, cost, and latency of this agentic AI application. 

In this session, we will cover below two things 
- [Built-in LLM Judges](https://docs.databricks.com/aws/en/generative-ai/agent-evaluation/llm-judge-reference)
- Global Guidelines

In [0]:
from mlflow.genai.scorers import (
    RetrievalGroundedness,
    RelevanceToQuery,
    Safety,
    Guidelines,
    Correctness,
    RetrievalRelevance,
    RetrievalSufficiency
)
from mlflow.entities import Feedback
from mlflow.genai.scorers import scorer
from mlflow.genai.judges import custom_prompt_judge


response_quality_prompt = """Evaluate the quality of this customer service response:

    <request>{{request}}</request>
    <response>{{response}}</response>

    Choose the most appropriate rating:

    [[excellent]]: Empathetic, complete solution, proactive help offered
    [[good]]: Addresses the issue adequately with professional tone
    [[poor]]: Incomplete, unprofessional, or misses key concerns"""

@scorer
def response_quality_scorer(inputs, outputs):
  feedbacks = []

  # Create a multi-level quality judge
  response_quality_judge = custom_prompt_judge(
      name="response_quality_scorer",
      prompt_template=response_quality_prompt,
      numeric_values={
              "excellent": 1.0,
              "good": 0.7,
              "poor": 0.1
          }
      )   
  
  #Evaluate the response quality 
  mlflow_metric_result = response_quality_judge(
        request=inputs,
        response=outputs
    )
  
  return Feedback(
        name="response_quality_scorer",
        value=mlflow_metric_result.value,
        rationale=mlflow_metric_result.rationale)
  
# Save the scorers as a variable so we can re-use them in step 7
finance_scorers = [
        Safety(),
        response_quality_scorer,
        RetrievalGroundedness(),  # Checks if email content is grounded in retrieved data
        RetrievalRelevance(),
        RetrievalSufficiency(),
        Guidelines(
            name="follows_instructions",
            guidelines="The generated content must follow the user_instructions in the request.",
        )
]

In [0]:
#Extract the traces from mlflow 
evaluation_traces = mlflow.search_traces(
                                       experiment_ids = ['64dda9588c12476e8dd745b33f8381e0'],
                                       run_id = '43c939ec006c4e8c9ab340694658008c'
                                       )

In [0]:
import mlflow 
itern = 2
with mlflow.start_run(run_name=f"finOps_assistant_offline_eval_{itern}") as run: 
  # Run evaluation
  eval_results = mlflow.genai.evaluate(
    data=evaluation_traces,     
    scorers= finance_scorers
    )